In [ ]:
# === auto-inserted: bev-solving src on path ===
import sys, pathlib
_root = pathlib.Path.cwd()
while _root != _root.parent and not (_root / 'src' / 'geometry.py').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


In [1]:
import os, copy, time, json, random, math
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

DATA_TRAIN = Path('/kaggle/input/datasets/letiti6e/bev-yandex/autonomy_yandex_dataset_train_kaggle_safe_v2/autonomy_yandex_dataset_train')
DATA_VAL   = Path('/kaggle/input/datasets/letiti6e/bev-yandex/autonomy_yandex_dataset_val_kaggle_safe/autonomy_yandex_dataset_val')
DATA_TEST  = Path('/kaggle/input/datasets/letiti6e/bev-yandex/autonomy_yandex_dataset_test_kaggle_safe/autonomy_yandex_dataset_test')
RTMDET_PRETRAIN_PATH = Path('/kaggle/input/models/letiti6e/rmdet/pytorch/default/1/rtmdet_l_8xb32-300e_coco_20220719_112030-5a0be7c4.pth')

cfg = {
    'run_dir': './runs/v4_rtmdet_pretrain',
    'img_hw': (384, 704),
    'batch_size': 8,
    'val_batch_size': 1,
    'grad_accum': 4,
    'epochs': 20,
    'warmup_epochs': 3,
    'lr_backbone': 3e-5,
    'lr_head': 3e-4,
    'weight_decay': 1e-4,
    'embed_weight_decay': 1e-2,
    'pos_weight': 5.0,
    'num_workers': 4,
    'val_num_workers': 0,
    'seed': 42,
    'ema_decay': 0.995,
    'val_target_size': 200,
    'min_rover_count': 30,
    'topk_rovers': 25,
    'rover_emb_dim': 8,
    'rover_cond_dim': 8,
    'mae_dedup_thr': 0.02,
    'dedup_camera': '/camera/inner/frontal/middle',
    'use_clean_cache': True,
    'use_dp_if_available': True,
    'pretrained_backbone_path': str(RTMDET_PRETRAIN_PATH),
    'post_scale_range': (0.90, 1.10),
    'rot_deg_range': (-3.0, 3.0),
    'flip_prob': 0.5,
    'drop_images_prob': 0.05,
    'max_dropped_images': 2,
}

RUN_DIR = Path(cfg['run_dir'])
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = RUN_DIR / 'preproc_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

random.seed(cfg['seed'])
np.random.seed(cfg['seed'])
torch.manual_seed(cfg['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_cuda = torch.cuda.device_count() if torch.cuda.is_available() else 0
print('device:', device)
print('cuda devices:', num_cuda)
if num_cuda:
    for i in range(num_cuda):
        print(i, torch.cuda.get_device_name(i))
print('pretrain path:', RTMDET_PRETRAIN_PATH)
print('pretrain exists:', RTMDET_PRETRAIN_PATH.exists())
print('train batch_size:', cfg['batch_size'], '| train workers:', cfg['num_workers'])
print('val batch_size:', cfg['val_batch_size'], '| val workers:', cfg['val_num_workers'])

with open(RUN_DIR / 'config.json', 'w') as f:
    json.dump({k: str(v) for k, v in cfg.items()}, f, indent=2)



device: cuda
cuda devices: 2
0 Tesla T4
1 Tesla T4
pretrain path: /kaggle/input/models/letiti6e/rmdet/pytorch/default/1/rtmdet_l_8xb32-300e_coco_20220719_112030-5a0be7c4.pth
pretrain exists: True
train batch_size: 8 | train workers: 4
val batch_size: 1 | val workers: 0


In [ ]:
from src.geometry import load_info_with_root, resolve_info_path, resolve_row_path

CAMERA_NAMES = [
    '/camera/inner/frontal/middle',
    '/camera/inner/frontal/far',
    '/side/left/forward',
    '/side/right/forward',
]
INTRINSICS_NAMES = [n + '/intrinsic_params' for n in CAMERA_NAMES]
CAR2CAM_NAMES = [n + '/car_to_cam' for n in CAMERA_NAMES]
GT_NAME = 'gt_occupancy_grid'

CAM_IDX = {name: i for i, name in enumerate(CAMERA_NAMES)}
LEFT_CAM = '/side/left/forward'
RIGHT_CAM = '/side/right/forward'
LEFT_IDX = CAM_IDX[LEFT_CAM]
RIGHT_IDX = CAM_IDX[RIGHT_CAM]

BEV_H, BEV_W = 188, 126
BEV_RES = 0.8
X_RANGE = (0.0, BEV_H * BEV_RES)
Y_RANGE = (-BEV_W * BEV_RES / 2, BEV_W * BEV_RES / 2)
Z_LEVELS = (0.3, 1.0, 2.0, 3.0)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.cleaning import build_img_hash, clean_merged_info, compute_gt_stats, smart_deduplicate


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.splits import build_rover_vocab_from_train, encode_rover, make_test_matched_split_target


In [5]:
clean_info, dedup_report, clean_summary = clean_merged_info(
    DATA_TRAIN,
    DATA_VAL,
    cache_dir=CACHE_DIR,
    mae_thr=cfg['mae_dedup_thr'],
    dedup_camera=cfg['dedup_camera'],
    use_cache=cfg['use_clean_cache'],
)
print(json.dumps(clean_summary, indent=2, ensure_ascii=False))
display(clean_info.head(3))
if len(dedup_report):
    display(dedup_report.head(10))

train_idx, val_idx = make_test_matched_split_target(
    clean_info,
    DATA_TEST / 'info.csv',
    target_val_size=cfg['val_target_size'],
    seed=cfg['seed'],
    cache_path=CACHE_DIR / 'test_matched_val200_split.npz',
)
print('train_idx:', len(train_idx), 'val_idx:', len(val_idx))

train_info = clean_info.iloc[train_idx].reset_index(drop=True).copy()
val_info = clean_info.iloc[val_idx].reset_index(drop=True).copy()

rover_vocab, rover_stats = build_rover_vocab_from_train(
    train_info,
    min_count=cfg['min_rover_count'],
    topk=cfg['topk_rovers'],
)
rover_stats.to_csv(RUN_DIR / 'rover_embedding_stats.csv', index=False)
print('num rover classes including Other:', len(rover_vocab))
print('vocab head:', list(rover_vocab.items())[:10])
display(rover_stats.head(30))


Smart dedup groups: 100%|██████████| 1473/1473 [02:06<00:00, 11.68it/s]


{
  "merged_before_clean": 5000,
  "removed_empty_gt": 117,
  "after_empty_filter": 4883,
  "removed_by_dedup": 390,
  "clean_total": 4493,
  "dedup_groups": 173
}


,gt_occupancy_grid,header_ts,log_time,message_ts,/camera/inner/frontal/middle,/side/left/forward,/side/right/forward,/camera/inner/frontal/far,/camera/inner/frontal/middle/intrinsic_params,/camera/inner/frontal/middle/car_to_cam,...,predicted_occupancy_grid,ride_date,ride_time,rover,scene_part_order,__data_root,__source_split,coverage,valid_count,pos_count
0,autonomy_yandex_dataset_train/static_grids/163...,1633857774533809000,12:18:58,1633857774533809000,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,...,autonomy_yandex_dataset_train/predicted_static...,2021-10-10,12:18:58,orvy,0,/kaggle/input/datasets/letiti6e/bev-yandex/aut...,train,0.061972,1468,602
1,autonomy_yandex_dataset_train/static_grids/163...,1636812143899937000,15:34:56,1636812143899937000,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,...,autonomy_yandex_dataset_train/predicted_static...,2021-11-13,15:34:56,shelly,0,/kaggle/input/datasets/letiti6e/bev-yandex/aut...,train,0.227077,5379,3752
2,autonomy_yandex_dataset_train/static_grids/163...,1633600207233930000,12:28:29,1633600207233930000,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,...,autonomy_yandex_dataset_train/predicted_static...,2021-10-07,12:28:29,orvy,0,/kaggle/input/datasets/letiti6e/bev-yandex/aut...,train,0.225008,5330,1264


,kept_row_id,group_size,members
0,2271,2,"[2271, 3683]"
1,4101,2,"[4473, 4101]"
2,1366,3,"[306, 2710, 1366]"
3,98,5,"[1721, 2885, 209, 3630, 98]"
4,4151,2,"[4824, 4151]"
5,1703,3,"[526, 277, 1703]"
6,4867,3,"[4332, 4815, 4867]"
7,2549,2,"[3137, 2549]"
8,2112,2,"[2112, 3216]"
9,2330,2,"[2938, 2330]"


train_idx: 4273 val_idx: 220
num rover classes including Other: 26
vocab head: [('__other__', 0), ('orvy', 1), ('shelly', 2), ('lerita', 3), ('ward', 4), ('ravine', 5), ('greben', 6), ('lucky', 7), ('miro', 8), ('benzon', 9)]


,rover,count,embedding_id,bucket
0,orvy,638,1,unique
1,shelly,496,2,unique
2,lerita,327,3,unique
3,ward,239,4,unique
4,ravine,208,5,unique
5,greben,194,6,unique
6,lucky,187,7,unique
7,miro,120,8,unique
8,benzon,114,9,unique
9,natelio,108,10,unique


In [6]:
class BEVDatasetV4RTMDet(Dataset):
    def __init__(self, info_df: pd.DataFrame, mode: str = 'train',
                 img_hw=(384, 704), aug: bool = False,
                 rover_vocab: dict | None = None,
                 post_scale_range=(0.90, 1.10),
                 rot_deg_range=(-3.0, 3.0),
                 flip_prob=0.5,
                 drop_images_prob=0.05,
                 max_dropped_images=2):
        self.info = info_df.reset_index(drop=True).copy()
        self.mode = mode
        self.img_hw = img_hw
        self.aug = aug and (mode == 'train')
        self.rover_vocab = rover_vocab or {'__other__': 0}
        self.normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
        self.post_scale_range = post_scale_range
        self.rot_deg_range = rot_deg_range
        self.flip_prob = flip_prob
        self.drop_images_prob = drop_images_prob
        self.max_dropped_images = max_dropped_images

    def __len__(self):
        return len(self.info)

    def _resolve_path(self, row: pd.Series, key: str) -> Path:
        return resolve_info_path(Path(row['__data_root']), row[key])

    def _letterbox(self, img: Image.Image):
        src_W, src_H = img.size
        H_t, W_t = self.img_hw
        s = min(W_t / src_W, H_t / src_H)
        new_W, new_H = int(round(src_W * s)), int(round(src_H * s))
        img_resized = img.resize((new_W, new_H), Image.BILINEAR)
        canvas = Image.new('RGB', (W_t, H_t), (0, 0, 0))
        pad_x = (W_t - new_W) // 2
        pad_y = (H_t - new_H) // 2
        canvas.paste(img_resized, (pad_x, pad_y))
        return canvas, s, pad_x, pad_y

    def _affine_params(self):
        if not self.aug:
            return 1.0, 0.0
        scale = random.uniform(*self.post_scale_range)
        rot_deg = random.uniform(*self.rot_deg_range)
        return scale, rot_deg

    def _apply_affine_canvas(self, canvas: Image.Image, scale: float, rot_deg: float):
        H_t, W_t = self.img_hw
        out = canvas
        if abs(scale - 1.0) > 1e-6:
            sW, sH = int(round(W_t * scale)), int(round(H_t * scale))
            scaled = out.resize((sW, sH), Image.BILINEAR)
            if scale >= 1.0:
                dx = random.randint(0, sW - W_t)
                dy = random.randint(0, sH - H_t)
                out = scaled.crop((dx, dy, dx + W_t, dy + H_t))
            else:
                out = Image.new('RGB', (W_t, H_t), (0, 0, 0))
                dx = (W_t - sW) // 2
                dy = (H_t - sH) // 2
                out.paste(scaled, (dx, dy))
        else:
            dx = 0
            dy = 0

        if abs(rot_deg) > 1e-6:
            out = out.rotate(rot_deg, resample=Image.BILINEAR, expand=False, fillcolor=(0, 0, 0))
        return out, dx, dy

    def _make_affine_matrix(self, scale: float, rot_deg: float, dx: float, dy: float):
        H_t, W_t = self.img_hw
        cx = (W_t - 1) / 2.0
        cy = (H_t - 1) / 2.0

        T1 = np.array([[1, 0, -cx], [0, 1, -cy], [0, 0, 1]], dtype=np.float32)
        S = np.array([[scale, 0, 0], [0, scale, 0], [0, 0, 1]], dtype=np.float32)
        T2 = np.array([[1, 0, cx - dx], [0, 1, cy - dy], [0, 0, 1]], dtype=np.float32)
        theta = np.deg2rad(rot_deg)
        c = np.cos(theta).astype(np.float32)
        s = np.sin(theta).astype(np.float32)
        R = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=np.float32)
        return T2 @ R @ S @ T1

    def _load_one_camera(self, row: pd.Series, cam_key: str, intr_key: str, c2c_key: str,
                         scale: float, rot_deg: float, flip_lr: bool):
        img = Image.open(self._resolve_path(row, cam_key)).convert('RGB')
        intr_path = self._resolve_path(row, intr_key)
        car2cam_path = self._resolve_path(row, c2c_key)

        canvas, s0, pad_x, pad_y = self._letterbox(img)
        canvas, dx, dy = self._apply_affine_canvas(canvas, scale, rot_deg)

        if flip_lr:
            canvas = canvas.transpose(Image.FLIP_LEFT_RIGHT)

        arr = np.array(canvas)
        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)
        img_t = torch.from_numpy(arr).permute(2, 0, 1).float() / 255.0
        img_t = self.normalize(img_t)

        intr_full = np.load(intr_path)
        K = intr_full[:, :3].copy().astype(np.float32)
        K[0, 0] *= s0; K[0, 2] *= s0
        K[1, 1] *= s0; K[1, 2] *= s0
        K[0, 2] += pad_x
        K[1, 2] += pad_y

        A = self._make_affine_matrix(scale, rot_deg, dx, dy)
        K = A @ K

        H_t, W_t = self.img_hw
        if flip_lr:
            Fm = np.array([[-1, 0, W_t - 1], [0, 1, 0], [0, 0, 1]], dtype=np.float32)
            K = Fm @ K

        car2cam = np.load(car2cam_path).astype(np.float32)
        return img_t, K, car2cam

    def _swap_lr_if_needed(self, items, flip_lr: bool):
        if not flip_lr:
            return items
        items = list(items)
        items[LEFT_IDX], items[RIGHT_IDX] = items[RIGHT_IDX], items[LEFT_IDX]
        return items

    def _drop_random_images(self, images):
        if not self.aug:
            return images
        if random.random() >= self.drop_images_prob:
            return images
        n_drop = random.randint(1, min(self.max_dropped_images, len(images)))
        drop_ids = random.sample(range(len(images)), k=n_drop)
        images = list(images)
        for di in drop_ids:
            images[di] = torch.zeros_like(images[di])
        return images

    def _load_sample(self, idx: int):
        row = self.info.iloc[idx]
        scale, rot_deg = self._affine_params()
        flip_lr = self.aug and (random.random() < self.flip_prob)

        imgs, Ks, M = [], [], []
        for cam, intr, c2c in zip(CAMERA_NAMES, INTRINSICS_NAMES, CAR2CAM_NAMES):
            img_t, K, c = self._load_one_camera(row, cam, intr, c2c, scale=scale, rot_deg=rot_deg, flip_lr=flip_lr)
            imgs.append(img_t)
            Ks.append(torch.from_numpy(K))
            M.append(torch.from_numpy(c))

        imgs = self._swap_lr_if_needed(imgs, flip_lr)
        Ks = self._swap_lr_if_needed(Ks, flip_lr)
        M = self._swap_lr_if_needed(M, flip_lr)
        imgs = self._drop_random_images(imgs)

        out = {
            'images': torch.stack(imgs, dim=0),
            'intrinsics': torch.stack(Ks, dim=0),
            'car2cams': torch.stack(M, dim=0),
            'rover_id': torch.tensor(encode_rover(row.get('rover', '__other__'), self.rover_vocab), dtype=torch.long),
            'info_idx': idx,
        }
        if self.mode == 'test':
            return out
        gt = np.load(self._resolve_path(row, GT_NAME)).squeeze()
        gt = np.where(gt < 0, 255, gt).astype(np.int64)
        out['gt'] = torch.from_numpy(gt).unsqueeze(0)
        return out

    def __getitem__(self, idx: int):
        max_tries = 5
        last_err = None
        for k in range(max_tries):
            try_idx = (idx + k) % len(self.info)
            try:
                return self._load_sample(try_idx)
            except (OSError, ValueError, FileNotFoundError) as e:
                last_err = e
                continue
        raise RuntimeError(f'Failed to load sample after {max_tries} tries from idx={idx}: {last_err}')



In [20]:
from src.models.cspnext import CSPLayer, CSPNeXtBackboneFromRTMDet, CSPNeXtBlock, ChannelAttention, ConvBNAct, SPPBottleneck, load_rtmdet_pretrained_backbone
from src.utils.training import extract_state_dict


class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_c, out_c, kernel_size=5, stride=1,
                 eps=1e-3, momentum=0.01):
        super().__init__()
        self.depthwise_conv = ConvBNAct(
            in_c, in_c, kernel_size,
            stride=stride,
            groups=in_c,
            eps=eps,
            momentum=momentum,
        )
        self.pointwise_conv = ConvBNAct(
            in_c, out_c, 1,
            stride=stride if False else 1,
            groups=1,
            eps=eps,
            momentum=momentum,
        )

    def forward(self, x):
        x = self.depthwise_conv(x)
        x = self.pointwise_conv(x)
        return x



In [ ]:
from src.models.decoder import SmallUNet, _UNetBlock

class MultiCamBEVv4RTMDet(nn.Module):
    def __init__(self, num_rover_classes: int,
                 rover_emb_dim: int = 8,
                 rover_cond_dim: int = 8,
                 n_cameras: int = 4,
                 freeze_backbone: bool = False,
                 pretrained_backbone_path: str | None = None):
        super().__init__()
        self.n_cameras = n_cameras
        self.bev_h, self.bev_w = BEV_H, BEV_W
        self.x_range, self.y_range = X_RANGE, Y_RANGE
        self.z_levels = Z_LEVELS
        self.rover_cond_dim = rover_cond_dim

        self.backbone = CSPNeXtBackboneFromRTMDet(
            arch='P5',
            deepen_factor=1.0,
            widen_factor=1.0,
            expand_ratio=0.5,
            channel_attention=True,
            out_indices=(2, 3, 4),
            eps=1e-3,
            momentum=0.01,
        )
        self.backbone_load_summary = None
        if pretrained_backbone_path:
            self.backbone_load_summary = load_rtmdet_pretrained_backbone(self.backbone, Path(pretrained_backbone_path))
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.backbone_proj = nn.Conv2d(1024, 128, 1)
        self.feat_proj = nn.Conv2d(128, 64, 1)
        self.register_buffer('ego_voxels', self._make_ego_voxels(), persistent=False)

        self.rover_embed = nn.Embedding(num_rover_classes, rover_emb_dim)
        nn.init.normal_(self.rover_embed.weight, std=0.02)
        self.rover_mlp = nn.Sequential(
            nn.Linear(rover_emb_dim, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, rover_cond_dim),
            nn.ReLU(inplace=True),
        )

        in_c = 64 * len(self.z_levels) + rover_cond_dim
        self.bev_decoder = SmallUNet(in_c=in_c, base_c=32, out_c=1)

    def _make_ego_voxels(self):
        xs = torch.linspace(self.x_range[0] + BEV_RES / 2, self.x_range[1] - BEV_RES / 2, self.bev_h)
        ys = torch.linspace(self.y_range[0] + BEV_RES / 2, self.y_range[1] - BEV_RES / 2, self.bev_w)
        zs = torch.tensor(self.z_levels, dtype=torch.float32)
        Z, X, Y = torch.meshgrid(zs, xs, ys, indexing='ij')
        ones = torch.ones_like(X)
        return torch.stack([X, Y, Z, ones], dim=-1)

    def forward(self, images, intrinsics, car2cams, rover_ids):
        B, N, C_, Hi, Wi = images.shape
        assert N == self.n_cameras

        feats = self.backbone(images.reshape(B * N, C_, Hi, Wi))
        feat = feats[-1]
        feat = self.backbone_proj(feat)
        feat = self.feat_proj(feat)
        Hf, Wf = feat.shape[-2:]
        feat = feat.reshape(B, N, 64, Hf, Wf)

        Z, H, W, _ = self.ego_voxels.shape
        V = Z * H * W
        voxels = self.ego_voxels.reshape(-1, 4).unsqueeze(0).unsqueeze(0).expand(B, N, -1, -1)

        p_cam = torch.einsum('bnij,bnvj->bniv', car2cams, voxels)
        p_cam_3d = p_cam[:, :, :3]
        uv_homog = torch.einsum('bnij,bnjv->bniv', intrinsics, p_cam_3d)
        z = uv_homog[:, :, 2]
        u = uv_homog[:, :, 0] / z.clamp(min=1e-3)
        v = uv_homog[:, :, 1] / z.clamp(min=1e-3)

        u_n = 2.0 * u / Wi - 1.0
        v_n = 2.0 * v / Hi - 1.0
        valid = (z > 0.1) & (u_n.abs() <= 1.0) & (v_n.abs() <= 1.0)

        grid = torch.stack([u_n, v_n], dim=-1).reshape(B * N, V, 1, 2)
        sampled = F.grid_sample(
            feat.reshape(B * N, 64, Hf, Wf),
            grid,
            mode='bilinear',
            padding_mode='zeros',
            align_corners=False,
        )
        sampled = sampled.squeeze(-1).reshape(B, N, 64, V)

        valid_f = valid.float().unsqueeze(2)
        sampled = sampled * valid_f
        agg = sampled.sum(dim=1) / valid_f.sum(dim=1).clamp(min=1.0)
        agg = agg.reshape(B, 64, Z, H, W).reshape(B, 64 * Z, H, W)

        rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
        rover_map = rover_feat.expand(-1, -1, H, W)
        agg = torch.cat([agg, rover_map], dim=1)
        return self.bev_decoder(agg)


In [ ]:
from src.eval import evaluate_iou
from src.losses import CompoundLossV2, _lovasz_grad, lovasz_hinge_flat
from src.metrics import iou_binary_batch, streaming_threshold_sweep
from src.utils.training import cleanup_cuda, unwrap_model, update_ema



In [23]:
ds_train_warm = BEVDatasetV4RTMDet(
    train_info,
    mode='train',
    img_hw=cfg['img_hw'],
    aug=False,
    rover_vocab=rover_vocab,
    post_scale_range=cfg['post_scale_range'],
    rot_deg_range=cfg['rot_deg_range'],
    flip_prob=cfg['flip_prob'],
    drop_images_prob=cfg['drop_images_prob'],
    max_dropped_images=cfg['max_dropped_images'],
)
ds_train_aug = BEVDatasetV4RTMDet(
    train_info,
    mode='train',
    img_hw=cfg['img_hw'],
    aug=True,
    rover_vocab=rover_vocab,
    post_scale_range=cfg['post_scale_range'],
    rot_deg_range=cfg['rot_deg_range'],
    flip_prob=cfg['flip_prob'],
    drop_images_prob=cfg['drop_images_prob'],
    max_dropped_images=cfg['max_dropped_images'],
)
ds_val = BEVDatasetV4RTMDet(
    val_info,
    mode='val',
    img_hw=cfg['img_hw'],
    aug=False,
    rover_vocab=rover_vocab,
    post_scale_range=cfg['post_scale_range'],
    rot_deg_range=cfg['rot_deg_range'],
    flip_prob=cfg['flip_prob'],
    drop_images_prob=cfg['drop_images_prob'],
    max_dropped_images=cfg['max_dropped_images'],
)

loader_warm = DataLoader(
    ds_train_warm,
    batch_size=cfg['batch_size'],
    shuffle=True,
    num_workers=cfg['num_workers'],
    pin_memory=(device.type == 'cuda'),
    drop_last=True,
)
loader_train = DataLoader(
    ds_train_aug,
    batch_size=cfg['batch_size'],
    shuffle=True,
    num_workers=cfg['num_workers'],
    pin_memory=(device.type == 'cuda'),
    drop_last=True,
)
loader_val = DataLoader(
    ds_val,
    batch_size=cfg['val_batch_size'],
    shuffle=False,
    num_workers=cfg['val_num_workers'],
    pin_memory=(device.type == 'cuda'),
)

print('loader_warm batch_size:', loader_warm.batch_size, '| workers:', loader_warm.num_workers)
print('loader_train batch_size:', loader_train.batch_size, '| workers:', loader_train.num_workers)
print('loader_val batch_size:', loader_val.batch_size, '| workers:', loader_val.num_workers)

sample = ds_train_aug[0]
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(k, tuple(v.shape), v.dtype)
    else:
        print(k, type(v), v)



loader_warm batch_size: 8 | workers: 4
loader_train batch_size: 8 | workers: 4
loader_val batch_size: 1 | workers: 0
images (4, 3, 384, 704) torch.float32
intrinsics (4, 3, 3) torch.float32
car2cams (4, 4, 4) torch.float32
rover_id () torch.int64
info_idx <class 'int'> 0
gt (1, 188, 126) torch.int64


In [24]:
!pip install mmengine

In [25]:
sanity_model = MultiCamBEVv4RTMDet(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=cfg['rover_emb_dim'],
    rover_cond_dim=cfg['rover_cond_dim'],
    pretrained_backbone_path=str(RTMDET_PRETRAIN_PATH),
).to(device)

with torch.no_grad():
    batch = next(iter(loader_warm))
    images = batch['images'].to(device)
    intr = batch['intrinsics'].to(device)
    c2c = batch['car2cams'].to(device)
    rover_id = batch['rover_id'].to(device)
    out = sanity_model(images, intr, c2c, rover_id)
    print('sanity output shape:', tuple(out.shape))

del sanity_model, out, batch, images, intr, c2c, rover_id
if device.type == 'cuda':
    torch.cuda.empty_cache()


{
  "checkpoint": "/kaggle/input/models/letiti6e/rmdet/pytorch/default/1/rtmdet_l_8xb32-300e_coco_20220719_112030-5a0be7c4.pth",
  "raw_keys": 874,
  "backbone_candidate_keys": 458,
  "loaded_keys": 458,
  "missing_keys": 0,
  "unexpected_keys": 0
}
sanity output shape: (8, 1, 188, 126)


In [26]:
base_model = MultiCamBEVv4RTMDet(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=cfg['rover_emb_dim'],
    rover_cond_dim=cfg['rover_cond_dim'],
    pretrained_backbone_path=str(RTMDET_PRETRAIN_PATH),
).to(device)

if cfg['use_dp_if_available'] and device.type == 'cuda' and torch.cuda.device_count() > 1:
    print('Wrapping model with DataParallel on', torch.cuda.device_count(), 'GPUs')
    model = nn.DataParallel(base_model)
else:
    model = base_model

core_model = unwrap_model(model)
backbone_params, embed_params, other_params = [], [], []
for name, p in core_model.named_parameters():
    if not p.requires_grad:
        continue
    if name.startswith('backbone.') or name.startswith('backbone_proj.'):
        backbone_params.append(p)
    elif name.startswith('rover_embed.'):
        embed_params.append(p)
    else:
        other_params.append(p)

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': cfg['lr_backbone'], 'weight_decay': cfg['weight_decay']},
    {'params': other_params, 'lr': cfg['lr_head'], 'weight_decay': cfg['weight_decay']},
    {'params': embed_params, 'lr': cfg['lr_head'], 'weight_decay': cfg['embed_weight_decay']},
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'], eta_min=1e-6)
loss_fn = CompoundLossV2(pos_weight=cfg['pos_weight']).to(device)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

ema_model = copy.deepcopy(core_model).to(device).eval()
for p in ema_model.parameters():
    p.requires_grad = False

print('trainable params M:', sum(p.numel() for p in core_model.parameters() if p.requires_grad) / 1e6)


{
  "checkpoint": "/kaggle/input/models/letiti6e/rmdet/pytorch/default/1/rtmdet_l_8xb32-300e_coco_20220719_112030-5a0be7c4.pth",
  "raw_keys": 874,
  "backbone_candidate_keys": 458,
  "loaded_keys": 458,
  "missing_keys": 0,
  "unexpected_keys": 0
}
Wrapping model with DataParallel on 2 GPUs
trainable params M: 28.243977


In [28]:
import gc

In [ ]:
log = []
best_iou = -1.0
best_ema_iou = -1.0
start = time.time()

for epoch in range(cfg['epochs']):
    model.train()
    loader = loader_warm if epoch < cfg['warmup_epochs'] else loader_train
    phase = 'WARM' if epoch < cfg['warmup_epochs'] else 'AUG'

    losses = []
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(loader, desc=f'ep{epoch:02d} [{phase}]')

    for step, batch in enumerate(pbar):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt'].to(device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(images, intr, c2c, rover_id)
            loss, parts = loss_fn(logits, gt)
            loss = loss / cfg['grad_accum']

        scaler.scale(loss).backward()

        if (step + 1) % cfg['grad_accum'] == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            update_ema(ema_model, model, cfg['ema_decay'])

        losses.append(loss.item() * cfg['grad_accum'])
        if step % 20 == 0:
            pbar.set_postfix(loss=f'{np.mean(losses[-50:]):.3f}', bce=f"{parts['bce']:.2f}")

    if len(loader) % cfg['grad_accum'] != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        update_ema(ema_model, model, cfg['ema_decay'])

    scheduler.step()

    cleanup_cuda()
    print('start val model @0.5')
    val_iou_05 = evaluate_iou(model, loader_val, threshold=0.5, desc=f'val model ep{epoch:02d}')

    cleanup_cuda()
    print('start val ema @0.5')
    val_iou_05_ema = evaluate_iou(ema_model, loader_val, threshold=0.5, desc=f'val ema ep{epoch:02d}')
    cleanup_cuda()

    elapsed = (time.time() - start) / 60

    row = {
        'epoch': epoch,
        'loss': float(np.mean(losses)),
        'val_iou_05': float(val_iou_05),
        'val_iou_05_ema': float(val_iou_05_ema),
        'minutes': float(elapsed),
    }

    print(
        f"ep{epoch:02d} | loss {np.mean(losses):.3f} | "
        f"val@0.5 {val_iou_05:.4f} | ema@0.5 {val_iou_05_ema:.4f} | "
        f"{elapsed:.1f}m"
    )

    if val_iou_05 > best_iou:
        best_iou = val_iou_05
        torch.save({
            'model_type': 'v4_rtmdet_pretrain',
            'model': unwrap_model(model).state_dict(),
            'epoch': epoch,
            'best_iou': float(val_iou_05),
            'best_t': 0.5,
            'rover_vocab': rover_vocab,
            'cfg': cfg,
        }, RUN_DIR / 'best.pt')
        print('  new best model:', val_iou_05)

    if val_iou_05_ema > best_ema_iou:
        best_ema_iou = val_iou_05_ema
        torch.save({
            'model_type': 'v4_rtmdet_pretrain',
            'model': unwrap_model(model).state_dict(),
            'ema': ema_model.state_dict(),
            'epoch': epoch,
            'best_ema_iou': float(val_iou_05_ema),
            'best_t_ema': 0.5,
            'rover_vocab': rover_vocab,
            'cfg': cfg,
        }, RUN_DIR / 'ema_best.pt')
        print('  new best ema:', val_iou_05_ema)

    log.append(row)
    pd.DataFrame(log).to_csv(RUN_DIR / 'log.csv', index=False)

    torch.save({
        'model_type': 'v4_rtmdet_pretrain',
        'model': unwrap_model(model).state_dict(),
        'ema': ema_model.state_dict(),
        'epoch': epoch,
        'rover_vocab': rover_vocab,
        'cfg': cfg,
    }, RUN_DIR / 'last.pt')



ep00 [WARM]: 100%|██████████| 534/534 [30:46<00:00,  3.46s/it, bce=1.11, loss=0.974]


start val model @0.5


start val ema @0.5


ep00 | loss 1.047 | val@0.5 0.4648 | ema@0.5 0.4209 | 32.4m
  new best model: 0.4647640383237114
  new best ema: 0.42086916450502815


ep01 [WARM]:  10%|▉         | 53/534 [03:08<28:19,  3.53s/it, bce=1.25, loss=0.972] 

In [ ]:
print(1)